In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import os

# Seed for reproducibility — same data every time you run
np.random.seed(42)
random.seed(42)

# ── CARRIERS ──────────────────────────────────────────────
# Each carrier has a different breach probability
# This makes our analysis interesting — some are clearly worse
CARRIERS = {
    "Delhivery":   0.12,   # Best performer
    "Shadowfax":   0.18,
    "Ecom Express":0.22,
    "Amazon Logs": 0.16,
    "DTDC":        0.28,   # Worst performer
    "Blue Dart":   0.14,
    "Xpressbees":  0.25,
}

# ── CITIES & PINCODES ─────────────────────────────────────
# Metro cities breach less (better infrastructure)
# Tier-2 and remote areas breach more
CITY_PINCODE_BREACH = {
    "Mumbai":    {"pincodes": ["400001","400051","400063","400080","400092"], "breach_multiplier": 0.8},
    "Delhi":     {"pincodes": ["110001","110011","110029","110045","110092"], "breach_multiplier": 0.85},
    "Bangalore": {"pincodes": ["560001","560034","560068","560078","560095"], "breach_multiplier": 0.82},
    "Hyderabad": {"pincodes": ["500001","500032","500072","500081","500090"], "breach_multiplier": 0.88},
    "Chennai":   {"pincodes": ["600001","600028","600040","600078","600096"], "breach_multiplier": 0.90},
    "Kolkata":   {"pincodes": ["700001","700019","700041","700064","700106"], "breach_multiplier": 0.95},
    "Pune":      {"pincodes": ["411001","411014","411028","411045","411057"], "breach_multiplier": 0.92},
    "Jaipur":    {"pincodes": ["302001","302012","302019","302033","302055"], "breach_multiplier": 1.10},
    "Lucknow":   {"pincodes": ["226001","226010","226020","226025","226030"], "breach_multiplier": 1.20},
    "Patna":     {"pincodes": ["800001","800009","800013","800020","800027"], "breach_multiplier": 1.45},
    "Bhopal":    {"pincodes": ["462001","462010","462016","462023","462038"], "breach_multiplier": 1.30},
    "Guwahati":  {"pincodes": ["781001","781005","781010","781021","781036"], "breach_multiplier": 1.50},
}

CATEGORIES = ["Electronics", "Clothing", "Books", "Home & Kitchen",
              "FMCG", "Pharmaceuticals", "Furniture", "Toys"]

BREACH_REASONS = ["Weather Delay", "Capacity Overload", "Incorrect Address",
                  "Customs Hold", "Vehicle Breakdown", "High Volume Period",
                  "Last Mile Issue", "Sorting Center Delay"]

ORIGIN_CITIES = ["Mumbai", "Delhi", "Bangalore", "Hyderabad", "Chennai"]

In [2]:
def generate_order(order_id, start_date):
    """
    Generates ONE realistic logistics order row.
    Called 50,000 times to build our full dataset.
    """

    # Random date within 2 years
    order_date = start_date + timedelta(days=random.randint(0, 729))

    # Pick carrier
    carrier = random.choice(list(CARRIERS.keys()))
    base_breach_prob = CARRIERS[carrier]

    # Pick destination city & pincode
    dest_city = random.choice(list(CITY_PINCODE_BREACH.keys()))
    pincode   = random.choice(CITY_PINCODE_BREACH[dest_city]["pincodes"])
    city_mult = CITY_PINCODE_BREACH[dest_city]["breach_multiplier"]

    # Pick origin (different from destination)
    origin = random.choice([c for c in ORIGIN_CITIES if c != dest_city] or ORIGIN_CITIES)

    # Category & weight
    category = random.choice(CATEGORIES)
    weight   = round(random.uniform(0.1, 25.0), 2)

    # Promised days — heavier/remote = more days promised
    if dest_city in ["Patna", "Guwahati", "Lucknow"]:
        promised_days = random.randint(4, 7)
    else:
        promised_days = random.randint(2, 5)

    # ── FESTIVE SEASON MULTIPLIER ──────────────────────────
    # October & November = Diwali season → more breaches
    month = order_date.month
    festive_mult = 1.4 if month in [10, 11] else 1.0
    # January = post-New Year rush
    festive_mult = 1.2 if month == 1 else festive_mult

    # ── FINAL BREACH PROBABILITY ──────────────────────────
    final_breach_prob = min(base_breach_prob * city_mult * festive_mult, 0.95)
    is_breach = random.random() < final_breach_prob

    # Actual days — if breach, exceeded promised
    if is_breach:
        delay = random.randint(1, 5)
        actual_days = promised_days + delay
        breach_reason = random.choice(BREACH_REASONS)
    else:
        actual_days = promised_days - random.randint(0, 1)
        actual_days = max(actual_days, 1)
        breach_reason = None

    # Shipment value
    value_map = {
        "Electronics": (5000, 80000), "Clothing": (300, 5000),
        "Books": (100, 1500),         "Home & Kitchen": (500, 15000),
        "FMCG": (100, 3000),          "Pharmaceuticals": (200, 8000),
        "Furniture": (3000, 50000),   "Toys": (200, 5000),
    }
    low, high = value_map[category]
    shipment_value = round(random.uniform(low, high), 2)

    return {
        "order_id":           f"ORD{order_id:07d}",
        "order_date":         order_date.strftime("%Y-%m-%d"),
        "carrier":            carrier,
        "origin_city":        origin,
        "destination_city":   dest_city,
        "destination_pincode":pincode,
        "category":           category,
        "shipment_weight_kg": weight,
        "promised_days":      promised_days,
        "actual_days":        actual_days,
        "is_breach":          int(is_breach),   # 1 or 0 (easier for SQL)
        "breach_reason":      breach_reason if breach_reason else "No Breach",
        "shipment_value_inr": shipment_value,
        "day_of_week":        order_date.strftime("%A"),
        "month":              order_date.strftime("%B"),
        "year":               order_date.year,
    }

In [3]:
def main():
    print("⏳ Generating 50,000 logistics orders...")

    start_date = datetime(2023, 1, 1)
    NUM_ORDERS = 50000

    orders = [generate_order(i+1, start_date) for i in range(NUM_ORDERS)]
    df = pd.DataFrame(orders)

    # ── SAVE TO CSV ───────────────────────────────────────
    os.makedirs("../data", exist_ok=True)
    output_path = "../data/logistics_data.csv"
    df.to_csv(output_path, index=False)

    # ── QUICK SANITY CHECK ────────────────────────────────
    print(f"\n✅ Data generated! Shape: {df.shape}")
    print(f"\n📊 Overall Breach Rate: {df['is_breach'].mean()*100:.1f}%")
    print(f"\n🚚 Breach Rate by Carrier:")
    print(df.groupby('carrier')['is_breach'].mean().sort_values(ascending=False).apply(lambda x: f"{x*100:.1f}%"))
    print(f"\n📍 Breach Rate by City:")
    print(df.groupby('destination_city')['is_breach'].mean().sort_values(ascending=False).apply(lambda x: f"{x*100:.1f}%"))
    print(f"\n📁 Saved to: {output_path}")

if __name__ == "__main__":
    main()

⏳ Generating 50,000 logistics orders...

✅ Data generated! Shape: (50000, 16)

📊 Overall Breach Rate: 22.1%

🚚 Breach Rate by Carrier:
carrier
DTDC            31.9%
Xpressbees      28.5%
Ecom Express    24.4%
Shadowfax       20.8%
Amazon Logs     18.5%
Blue Dart       16.8%
Delhivery       14.1%
Name: is_breach, dtype: object

📍 Breach Rate by City:
destination_city
Guwahati     31.6%
Patna        30.9%
Bhopal       26.5%
Lucknow      25.2%
Jaipur       23.5%
Kolkata      20.3%
Pune         19.0%
Chennai      18.6%
Delhi        18.5%
Hyderabad    17.7%
Bangalore    17.3%
Mumbai       16.5%
Name: is_breach, dtype: object

📁 Saved to: ../data/logistics_data.csv
